# Importation des bibliothèques

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pathlib import Path

# Constantes

In [ ]:
# chemin pour accéder aux données
FILE_PATH = '../src/data/clean/donnees_immobilieres_cleaned.delta'

# chemin pour sauvegarder les résultats
CLASSIFICATION_OUTPUT_PATH = "../src/data/ml/classification/ministere_predictions.delta"
CLUSTERING_OUTPUT_PATH = "../src/data/ml/clustering/clusters.delta"

# Application

In [ ]:
# Configuer sparksession pour utiliser delta lake
builder = (
    SparkSession.builder
    .appName("RealEstateML")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# Charger les données
df = spark.read.format("delta").load(FILE_PATH)

In [ ]:
# Afficher le schéma du DataFrame
df.printSchema()

In [ ]:
# Afficher les 5 premières lignes du DataFrame
df.show(5)

## Classification

In [ ]:
# Sélection des colonnes pertinentes et suppression des lignes avec des valeurs manquantes
df_ml = df.select("type", "fonction", "region", "dept", "ministere").dropna()

In [ ]:
# Indexer pour les variables catégorielles
categorical_cols = ["type", "fonction", "region", "dept"]
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_index")
    for col in categorical_cols
]

In [ ]:
# Indexer pour la variable cible
label_indexer = StringIndexer(inputCol="ministere", outputCol="label")

In [ ]:
# OneHotEncoder pour les variables catégorielles
encoder = OneHotEncoder(
    inputCols=[f"{col}_index" for col in categorical_cols],
    outputCols=[f"{col}_vec" for col in categorical_cols]
)

In [ ]:
# Assemblage des features
assembler = VectorAssembler(
    inputCols=[f"{col}_vec" for col in categorical_cols],
    outputCol="features"
)

In [ ]:
# Création du modèle de classification
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)

In [ ]:
# Création du pipeline
pipeline = Pipeline(stages=indexers + [label_indexer, encoder, assembler, rf])

In [ ]:
# Split des données en train et test
train, test = df_ml.randomSplit([0.8, 0.2], seed=29)

In [ ]:
# Entraînement du modèle
model = pipeline.fit(train)

In [ ]:
# Prédictions sur le jeu de test
predictions = model.transform(test)

In [ ]:
# Calcul de l'accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

In [ ]:
# Calcul de l'accuracy
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy : {accuracy}")

In [ ]:
# Création du dossier si il n'existe pas
Path(CLASSIFICATION_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Sélection des colonnes à sauvegarder
predictions = predictions.select(
    "type",
    "region",
    "dept",
    "ministere",
    "prediction",
    "probability"
)

In [ ]:
# Sauvegarde des prédictions de classification
predictions.write.format("delta").mode("overwrite").save(CLASSIFICATION_OUTPUT_PATH)

## Clustering

In [ ]:
# Sélection des colonnes pertinentes pour le clustering et suppression des lignes avec des valeurs manquantes
df_cluster = df.select("type", "fonction", "region", "dept", "ministere").dropna()

In [ ]:
# Indexer pour les variables catégorielles
categorical_cols = ["type", "fonction", "region", "dept", "ministere"]

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_index")
    for col in categorical_cols
]

In [ ]:
# OneHotEncoder pour les variables catégorielles
encoder = OneHotEncoder(
    inputCols=[f"{col}_index" for col in categorical_cols],
    outputCols=[f"{col}_vec" for col in categorical_cols]
)

In [ ]:
# Assemblage des features
assembler = VectorAssembler(
    inputCols=[f"{col}_vec" for col in categorical_cols],
    outputCol="features"
)

In [ ]:
# Création du modèle de clustering
kmeans = KMeans(k=5, seed=29)

In [ ]:
# Création du pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler, kmeans])

In [ ]:
# Entraînement du modèle de clustering
model = pipeline.fit(df_cluster)

In [ ]:
# Prédictions de clustering
cluster_predictions = model.transform(df_cluster)

In [ ]:
# Affichage des prédictions de clustering
cluster_predictions.select("type", "region", "prediction").show()

In [ ]:
# Création du dossier si il n'existe pas
cluster_predictions = cluster_predictions.select(
    "type",
    "region",
    "dept",
    "ministere",
    "prediction"  
)

In [ ]:
# Sauvegarde des prédictions de clustering
Path(CLUSTERING_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Sauvegarde des prédictions de clustering
cluster_predictions.write.format("delta").mode("overwrite").save(CLUSTERING_OUTPUT_PATH)